[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Python from the Start](https://johnfisher-ai.github.io/Python-Visual-Guides/python-from-the-start.html)

# Environments and pip


## What you will be able to do

Install a package, find out which version you have, and explain why code that runs on one
machine fails on another. You will also be able to say what a virtual environment is and why
every real project has one.


## The idea

### The problem

The **Modules and Imports** notebook covered two kinds of code: the code you write, and the
code that ships with Python. Most of what you will actually use falls into a third kind,
written by other people and installed separately.

`numpy`, `pandas`, `matplotlib`, `requests`, `scikit-learn`: none come with Python. Someone else
wrote them, published them, and you install them. The **NumPy** guide onward depends entirely on
this, and so does most work anyone actually gets paid for.

Two problems follow immediately.

**Finding and installing them.** That part is easy, and is one command.

**Keeping them straight.** This is the hard part and the reason this notebook exists. Different
projects need different versions of the same package. Code written against pandas 1.5 can break
on pandas 2.0. A tutorial from 2021 assumes what was current in 2021. Install everything into
one shared place and every project fights over it; the last install wins and something else
quietly stops working.

### What pip and an environment are

> **PyPI** is the public index of Python packages. **pip** is the tool that downloads from it
> and installs into your Python.
>
> A **virtual environment** is a private folder holding its own copy of Python and its own
> installed packages. Activating one makes `python` and `pip` refer to that copy, so what you
> install for one project cannot affect another.

The mental model worth carrying: without an environment there is one shared shelf and every
project takes from it. With environments, each project gets its own shelf, and a version on one
shelf is invisible to the others.

### Why the tutorial broke

That phrase is the notebook's subtitle and it is worth being concrete about.

You follow a tutorial, run the code, and get an error the tutorial does not mention. Nothing is
wrong with your typing. The tutorial was written against a different version of a package, and
in the meantime a function was renamed, an argument was removed, or a default changed.

Three things tell you when this has happened: the error names something that should exist, the
same code works in someone else's notebook, or the tutorial is more than a year or two old.
The fix is to find out what version you have, and what version it expected.

### Where this fits with Colab

Colab arrives with a large set of packages already installed, which is why the notebooks in this
guide have needed no installation at all. When you need something it does not have, you install
it from inside the notebook, and that install lasts only until the session ends.

Colab is a convenience. It is not how a project is set up on your own machine, and this notebook
covers both.

### What this notebook covers

- Where a package comes from, and how to tell stdlib from installed
- `pip install`, and the Colab form of it
- Finding the version you actually have, three ways
- The difference between a module name and a package name
- Virtual environments: what they do, and the commands that cover ordinary use
- Where an environment lives, and how several of them coexist
- `requirements.txt`, and why a project has one
- Three errors, including the one behind most broken tutorials

### A first look

Before any of the detail, here is the whole idea. There is nothing to run yet: read it, and read
the output underneath it. Everything from Setup onward is where you start running things.

In a terminal, on your own machine:

```
python -m venv project-env           # 'venv' is the module; 'project-env' is a name you pick
source project-env/bin/activate      # start using it
pip install pandas                   # installs into project-env only
pip freeze > requirements.txt        # record exactly what is installed
```

`project-env` is a folder name of your choosing. Most projects call it `.venv`, and the Worked
examples explain why.

Inside a Colab notebook, where there is no terminal:

```python
!pip install pandas
```

And from Python, to find out what you actually have:

```python
import importlib.metadata as md

print(md.version("pip"))
```

```
25.3
```

The version you see will not be the one printed here, and that difference is the entire subject
of this notebook.


## Setup

Two imports from the standard library. Nothing is installed by this notebook.

**Run this cell before the rest of the notebook.**


In [1]:
import sys
import importlib.metadata as md
import importlib.util

print("Python", sys.version.split()[0])


Python 3.14.2


## Worked examples

### Where does this module come from

Some modules ship with Python and some were installed. Both are imported the same way, and the
file location tells you which is which.


In [2]:
import re
import json

for module in (re, json):
    where = module.__file__ or ""
    kind = "installed package" if "site-packages" in where else "standard library"
    print(f"{module.__name__:<8} {kind}")


re       standard library
json     standard library


Anything under `site-packages` was installed by `pip`. Everything else came with Python.

The distinction matters when something is missing. A standard library module is always there. An
installed one has to be installed on every machine that runs your code, which is what
`requirements.txt` later in this notebook is for.


### Installing something

One command, run in a terminal:

```
pip install requests
```

Colab has no terminal, so a notebook cell can run shell commands by starting the line with `!`:

```python
!pip install requests
```

Neither is run in this notebook, because installing packages during a build would be slow and
would change the environment for everything after it. The forms are worth recognizing rather
than running here.

Two details that save confusion later:

- `pip install requests` installs the **latest** version. `pip install "pandas==2.0.3"` pins an
  exact one, and `pip install "pandas>=2.0"` sets a floor.
- In Colab, an install lasts only until the session ends. Reopen the notebook tomorrow and it is
  gone. Put the install in a cell at the top so it runs with everything else.

### Which version do I have

Three ways, and they answer slightly different questions.


In [3]:
import importlib.metadata as md

print("pip:", md.version("pip"))


pip: 25.3


`importlib.metadata.version` asks the installation records, and works for anything installed by
`pip`. This is the one to reach for.


In [4]:
import json

print("json module reports:", getattr(json, "__version__", "no __version__ attribute"))


json module reports: 2.0.9


Many modules carry a `__version__` attribute, and many do not. It is convenient when present and
cannot be relied on.

The third way is `pip show pandas` in a terminal, or `!pip show pandas` in Colab, which prints
the version along with where it was installed and what it depends on.

To check whether something is available at all, without importing it and without raising:


In [5]:
import importlib.util

for name in ["json", "numpy", "some_package_nobody_has"]:
    available = importlib.util.find_spec(name) is not None
    print(f"{name:<26} available: {available}")


json                       available: True
numpy                      available: True
some_package_nobody_has    available: False


### Module names and package names are not always the same

You install one name and import another, more often than you would expect.

| You install | You import |
|---|---|
| `pip install scikit-learn` | `import sklearn` |
| `pip install pillow` | `import PIL` |
| `pip install beautifulsoup4` | `import bs4` |
| `pip install python-dateutil` | `import dateutil` |

This is a common source of "but I installed it". The install name is what PyPI calls the
project; the import name is whatever the authors chose for the folder. When they differ, the
package's own documentation says so on the first page.

It also explains an error worth recognizing: `md.version()` takes the **install** name, so
asking it about a standard library module fails even though the import works.


In [6]:
import re

print("import re works fine")
print(md.version("re"))


import re works fine


PackageNotFoundError: No package metadata was found for re

`No package metadata was found for re` is correct and slightly confusing. `re` is part of
Python, so nothing installed it and there are no installation records to look up.


### Virtual environments

On your own machine, this is the part that matters.

One thing to separate first, because the usual way of writing these commands hides it. In
`python -m venv .venv` the word appears twice and means something different each time:

- `venv` is the **module** that builds environments. It ships with Python and the name is fixed.
- `.venv` is the **folder name you chose**. Nothing requires it. It could be `project-env`,
  `myVirtualEnv`, or anything else.

Below, the folder is called `project-env` so the two never look alike. The two `activate` lines
are alternatives, one for each kind of system, so this is four steps:

```
python -m venv project-env             # create the folder, once per project
source project-env/bin/activate        # on macOS and Linux
project-env\Scripts\activate           # on Windows
pip install pandas                     # now installs into project-env, not system-wide
deactivate                             # stop using it
```

The only name you chose is the one following `venv`. Every later `project-env` is just that
folder being referred to again, which is why they all change together if you rename it. The
activate script lives inside the folder, so its path always follows the name.

**In practice, almost every project calls it `.venv`**, and the same steps then read:

```
python -m venv .venv
source .venv/bin/activate
pip install pandas
deactivate
```

The leading dot hides the folder from a plain `ls`, anyone opening your project recognizes it
without being told, and one line in `.gitignore` covers it everywhere. You are free to choose
another name; the cost is that every tutorial, editor and CI configuration you meet assumes
`.venv`.

### If they are all called .venv, how do you have more than one

This is the question the convention raises, and the answer is the piece that has been missing so
far: **the environment lives inside the project folder.** It is not stored centrally and it is
not registered anywhere. It is just a folder sitting next to your code.

```
~/projects/
    analysis/
        clean.py
        requirements.txt
        .venv/          <- this project's packages
    scraper/
        fetch.py
        requirements.txt
        .venv/          <- a completely separate set
```

Two environments, both called `.venv`, and no conflict at all. The name only has to be unique
inside its own folder, in exactly the way both projects can have a `README.md`.

You switch between them by activating the one belonging to the project you are working on:

```
cd ~/projects/analysis
source .venv/bin/activate      # now pip installs into the analysis environment

deactivate
cd ~/projects/scraper
source .venv/bin/activate      # now pip installs into the scraper environment
```

Only one is active in a shell at a time. Activating a second one in the same shell replaces the
first rather than combining them, which is what you want: the whole purpose is that `pandas`
version 1.5 in one project cannot reach the project that needs 2.2.

Two habits follow from this:

- **One environment per project, created in the project folder.** Not one per machine, and not
  one shared between related projects.
- **`cd` to the project first, then activate.** Almost every mistake here is running `pip
  install` in a shell where a different environment, or none at all, is active.

If you lose track of which one is active, the check at the end of this section tells you, and
most shells show the environment name in the prompt while one is active.

Activating changes what the words `python` and `pip` run. Before you activate, typing `pip` runs
the one installed on your system. After you activate, the same typed word runs the copy inside
the environment folder instead, and it installs there.

Nothing is moved or hidden. The shell is simply looking in a different folder first, for as long
as the environment is active. Anything you install goes into that folder and is invisible to
every other project, and deleting the folder removes the environment with no trace left behind.

Never commit the environment folder itself. It holds thousands of files, it is specific to your
machine, and it can always be rebuilt from a short list, which is what the next section is
about.

You can see which Python is currently in charge:


In [7]:
import sys

print("running from a folder ending:", "/".join(sys.executable.split("/")[-3:]))
print("inside a virtual environment:", sys.prefix != sys.base_prefix)


running from a folder ending: 3.14/bin/python3
inside a virtual environment: False


`sys.prefix` is the Python being used and `sys.base_prefix` is the one it was made from. They
differ inside a virtual environment and match outside one, which is the quickest way to check.

Colab does not use one, which is why the answer above is `False`.


### requirements.txt

An environment is only useful if someone else can reproduce it.

```
pip freeze > requirements.txt     # write down exactly what is installed
pip install -r requirements.txt   # install exactly that, somewhere else
```

`pip freeze` lists every installed package with its exact version:

```
numpy==1.26.4
pandas==2.2.1
python-dateutil==2.9.0
```

Commit that file. Anyone who clones the project runs the second command and gets the same
versions you had, which is the difference between "works on my machine" and "works".

This is also the answer to the broken tutorial. A tutorial that ships a `requirements.txt`
still works years later, because it says exactly what it was written against.

The **Testing and Packaging** guide covers the modern alternatives, which record the same
information in a different file.


### How many packages are here


In [8]:
names = sorted({d.metadata["Name"] for d in md.distributions() if d.metadata["Name"]})

print(len(names), "packages installed in this environment")
print("a few of them:", names[:5])


178 packages installed in this environment
a few of them: ['ImageIO', 'Jinja2', 'MarkupSafe', 'PyMuPDF', 'PyYAML']


That number will not match the one saved in this notebook, and neither will the names. It
depends entirely on where the notebook is running, which is the point.

On a fresh virtual environment the count is two or three. Colab arrives with several hundred,
which is why so much works there with no installation at all.


## Your turn

Six tasks. Write your answer in the cell under each and run it.

Nothing here installs anything. Two tasks ask you to write a command rather than run it.

Try each one before you look at an answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/python-from-the-start/19-environments-and-pip-solutions.ipynb).

**1.** Print the version of Python you are running, and the version of `pip`.


In [9]:
# your code here


**2.** For `csv`, `sqlite3` and one package of your choosing, print whether each is standard
library or an installed package, using `__file__`.


In [10]:
# your code here


**3.** Using `find_spec`, print whether `pandas` and `nosuchlibrary` are available, without
importing either.


In [11]:
# your code here


**4.** In a comment, write the two commands that create a virtual environment and activate it on
macOS or Linux.


In [12]:
# your code here


**5.** In a comment, write the command that records the current packages into
`requirements.txt`, and the one that installs from it.


In [13]:
# your code here


**6.** Print whether you are currently inside a virtual environment.


In [14]:
# your code here


## Common errors

Each cell below is run on purpose so you can see the real message.

### ModuleNotFoundError: it was never installed

The same error the **Modules and Imports** notebook covered, now with its most common cause.


In [15]:
import some_library_that_does_not_exist


ModuleNotFoundError: No module named 'some_library_that_does_not_exist'

When the name is spelled correctly, this means the package is not installed **in the Python that
is running**. That last part matters: it is possible to install into one Python and run another,
which is exactly what virtual environments exist to make explicit.

If `pip install x` succeeded and `import x` still fails, check that the two are the same Python:


In [16]:
print("this notebook is running:", "/".join(sys.executable.split("/")[-2:]))


this notebook is running: bin/python3


In a terminal, `which python` and `which pip` should point into the same place.


### PackageNotFoundError: asking about the wrong name


In [17]:
print(md.version("sklearn"))


PackageNotFoundError: No package metadata was found for sklearn

`No package metadata was found for sklearn`, and yet `import sklearn` works wherever it is
installed. The install name is `scikit-learn`; `sklearn` is only the import name.

`md.version("scikit-learn")` is the question that has an answer.


### The quiet one: the version you did not check

This does not raise. It gives you a different answer from the tutorial.


In [18]:
import json

records = json.loads('{"b": 1, "a": 2}')

print(list(records))


['b', 'a']


That output depends on behavior that has changed across Python versions. Dictionary ordering was
not guaranteed before Python 3.7, so the same three lines gave a different order on an older
Python, and code that relied on it broke silently when it moved.

Nothing here is wrong. The lesson is that **the version is part of the answer**. When output
does not match what a tutorial shows and the code is identical, compare versions before
comparing anything else:

```python
import sys
print(sys.version)
print(md.version("the-package-in-question"))
```


## Recap

- `pip install x` downloads from PyPI; in Colab the same thing is `!pip install x`.
- A file under `site-packages` was installed; anything else is the standard library.
- `importlib.metadata.version("name")` reports what you actually have, using the **install** name.
- Install names and import names differ often: `scikit-learn` gives you `sklearn`.
- A **virtual environment** gives one project its own packages, so versions cannot collide.
- It lives **inside the project folder**, so every project can have a `.venv` of its own
  and only the one you activate is in use.
- `sys.prefix != sys.base_prefix` tells you whether you are inside one.
- `pip freeze > requirements.txt` records the versions, and `pip install -r` reproduces them.
- A Colab install lasts only for the session.
- When code matches a tutorial and the output does not, compare versions first.


## What is next

The **Debugging** notebook, which is about what to do when something is wrong and the error does
not tell you enough: reading state, narrowing down where the problem is, and the tools that beat
adding `print` everywhere.


---

&#8592; **Previous:** [Modules and Imports](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/python-from-the-start/18-modules-and-imports.ipynb)  &nbsp;·&nbsp;  [Python from the Start Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/python-from-the-start.html)
